### Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\prana\AppData\Local\Temp\ipykernel_11532\885881214.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
c:\.py_notebook\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory safely and efficiently."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Check if the directory actually exists
    if not pdf_dir.exists():
        print(f"✗ Error: The directory '{pdf_directory}' does not exist.")
        return []
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            # PyPDFLoader prefers the resolved absolute string path
            loader = PyPDFLoader(str(pdf_file.resolve()))
            documents = loader.load()
            
            if not documents:
                print(f"  ⚠ Warning: No text or pages could be extracted from {pdf_file.name}")
                continue
            
            # Efficiently add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error loading {pdf_file.name}: {e}")
    
    print(f"\nTotal structural pages loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 5 PDF files to process

Processing: Artificial_intelligence.pdf
  ✓ Loaded 8 pages

Processing: Computer_vision.pdf
  ✓ Loaded 11 pages

Processing: Deep_learning.pdf
  ✓ Loaded 16 pages

Processing: Machine_learning.pdf
  ✓ Loaded 12 pages

Processing: Natural_language_processing.pdf
  ✓ Loaded 11 pages

Total structural pages loaded: 58


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'www.smallpdf.com', 'creator': 'www.smallpdf.com', 'creationdate': 'D:20260915172248', 'moddate': 'D:20260915172248', 'title': 'file.pdf', 'source': 'C:\\.py_notebook\\rag\\data\\pdf\\Artificial_intelligence.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='Artificial intelligence (AI) is the capability of computational systems to\nperform tasks typically associated with human intelligence, such as learning,\nreasoning, problem-solving, perception, and decision-making. It is a field of\nresearch in engineering, mathematics, and computer science that develops and\nstudies methods and software that enable machines to perceive their\nenvironment and use learning and intelligence to take actions that maximise\ntheir chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines,\nchatbots, virtual assistants, autonomous vehicles, pla

In [4]:
###Text Splitting

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show examples of a chunk
    if split_docs:
        print("\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)

Split 58 documents into 259 chunks

Example chunk:
Content: Artificial intelligence (AI) is the capability of computational systems to
perform tasks typically associated with human intelligence, such as learning,
reasoning, problem-solving, perception, and dec...
Metadata: {'producer': 'www.smallpdf.com', 'creator': 'www.smallpdf.com', 'creationdate': 'D:20260915172248', 'moddate': 'D:20260915172248', 'title': 'file.pdf', 'source': 'C:\\.py_notebook\\rag\\data\\pdf\\Artificial_intelligence.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_intelligence.pdf', 'file_type': 'pdf'}


### embedding and vector 

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple 
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
### Initiaize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1fa68f7e-0aa9-4f9c-83c0-0bbec87f5acb)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./sentence_bert_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: f7ceb545-29d6-46e3-b412-29d852b556c0)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./sentence_bert_config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: dee9561a-0289-4341-a0a0-030f0520a520)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./sentence_bert_config.json
Retrying in 4s [Retry 3/5].


Model loaded successfully. Embedding dimension: 384


C:\Users\prana\AppData\Local\Temp\ipykernel_11532\3102907331.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vectorstore

In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 259


In [9]:
chunks

[Document(metadata={'producer': 'www.smallpdf.com', 'creator': 'www.smallpdf.com', 'creationdate': 'D:20260915172248', 'moddate': 'D:20260915172248', 'title': 'file.pdf', 'source': 'C:\\.py_notebook\\rag\\data\\pdf\\Artificial_intelligence.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1', 'source_file': 'Artificial_intelligence.pdf', 'file_type': 'pdf'}, page_content='Artificial intelligence (AI) is the capability of computational systems to\nperform tasks typically associated with human intelligence, such as learning,\nreasoning, problem-solving, perception, and decision-making. It is a field of\nresearch in engineering, mathematics, and computer science that develops and\nstudies methods and software that enable machines to perceive their\nenvironment and use learning and intelligence to take actions that maximise\ntheir chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines,\nchatbots, virtual assistants, autonomous vehicles, pla

In [10]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks] 

## Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)

## store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 259 texts...


Batches: 100%|██████████| 9/9 [00:18<00:00,  2.06s/it]


Generated embeddings with shape: (259, 384)
Adding 259 documents to vector store...
Successfully added 259 documents to vector store
Total documents in collection: 518


### Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)        

In [13]:
rag_retriever

In [15]:
rag_retriever.retrieve(f"What is machine learning")

Retrieving documents for query: 'What is machine learning'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 47.39it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_89cb760a_161',
  'content': 'Machine learning (ML) is a field of study in artificial intelligence\nconcerned with the development and study of statistical algorithms that can\nlearn from data and generalize to unseen data, and thus perform tasks\nwithout being explicitly programmed.\nStatistics and mathematical optimisation methods compose the foundations of\nmachine learning. Data mining is a related field of study, focusing on\nexploratory data analysis (EDA) through unsupervised learning.\nFrom a theoretical viewpoint, probably approximately correct learning\nprovides a mathematical and statistical framework for describing machine\nlearning. Most traditional machine learning and deep learning algorithms can\nbe described as empirical risk minimisation under this framework.\nAdvances in the field of deep learning have allowed neural networks, a class\nof statistical algorithms, to surpass many previous machine learning\napproaches in performance. The resulting dominance 